# Warsaw Data Job Market Analysis
 
## Objective
 
Analyse data-related job opportunities in Warsaw to identify
in-demand technical skills, common job roles, experience
requirements and employment patterns.
 
## Business Questions
 
1. What data-related roles appear in the dataset?
2. Which technical skills appear most frequently?
3. How does skill demand differ between job roles?
4. Which skills commonly occur together?
5. What technologies appear in junior-level positions?
6. What are the most common working arrangements?
7. What insights can an entry-level candidate derive from the data?

In [3]:
import pandas as pd
import random
import os

# 1. Automatically build the data folder structure
os.makedirs('data/raw', exist_ok=True)

# 2. Define realistic Warsaw IT data market variables for 2026
roles = ['Data Analyst', 'Data Engineer', 'Data Scientist', 'BI Developer', 'Machine Learning Engineer']
companies = ['Allegro', 'mBank', 'PKO BP', 'CD Projekt Red', 'Sabre', 'Capgemini', 'EY', 'Deloitte', 'Samsung Warsaw']
tech_pool = ['Python', 'SQL', 'Tableau', 'Power BI', 'AWS', 'Azure', 'Spark', 'Kafka', 'Docker', 'Excel']
modes = ['Remote', 'Hybrid', 'On-site']
contracts = ['B2B', 'Permanent']

# 3. Generate 800 rows of raw data mimicking web-scraped job listings
raw_records = []
for i in range(800):
    job_role = random.choices(roles, weights=[0.35, 0.25, 0.20, 0.15, 0.05])[0]
    skills = random.sample(tech_pool, k=random.randint(2, 5))
    mode = random.choices(modes, weights=[0.40, 0.50, 0.10])[0]
    
    # Intentionally messy salary formats to practice data cleaning
    if random.random() < 0.15:
        salary = "Undisclosed"
    else:
        min_sal = random.randint(8, 14) if job_role in ['Data Analyst', 'BI Developer'] else random.randint(15, 26)
        salary = f"{min_sal}000 - {min_sal + random.randint(3, 7)}000 PLN"

    raw_records.append({
        'title': job_role,
        'company_name': random.choice(companies),
        'location': 'Warsaw' if mode != 'Remote' else 'Remote',
        'work_mode': mode,
        'salary_range': salary,
        'main_technology': skills[0],
        'all_skills': ", ".join(skills),
        'contract_type': random.choice(contracts)
    })

# 4. Save to your local project folder
df_local = pd.DataFrame(raw_records)
df_local.to_csv('data/raw/raw_warsaw_job_market.csv', index=False)

print("✅ Success! Local dataset created without needing an internet link.")
print(f"📊 Dataset contains {df_local.shape[0]} rows and is saved inside data/raw/.")


✅ Success! Local dataset created without needing an internet link.
📊 Dataset contains 800 rows and is saved inside data/raw/.


In [4]:
# Load the dataset we just created
df = pd.read_csv('data/raw/raw_warsaw_job_market.csv')

# Look at the first 5 rows of data
df.head()


,title,company_name,location,work_mode,salary_range,main_technology,all_skills,contract_type
0,Data Analyst,mBank,Warsaw,On-site,8000 - 14000 PLN,Excel,"Excel, Kafka, Python, Spark",Permanent
1,BI Developer,PKO BP,Remote,Remote,8000 - 13000 PLN,Spark,"Spark, Kafka",Permanent
2,Machine Learning Engineer,CD Projekt Red,Warsaw,Hybrid,26000 - 32000 PLN,Excel,"Excel, AWS",B2B
3,Data Engineer,Samsung Warsaw,Remote,Remote,26000 - 30000 PLN,AWS,"AWS, Docker, Azure",B2B
4,BI Developer,mBank,Warsaw,Hybrid,11000 - 18000 PLN,Azure,"Azure, Spark",Permanent


In [5]:
# Check the total dimensions of your data structure (Rows, Columns)
df.shape


(800, 8)

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- Question 1: What data-related roles appear in the dataset? ---
print("📊 --- QUESTION 1: JOB ROLE DISTRIBUTION ---")
role_counts = df['title'].value_counts()
print(role_counts)
print("\n" + "="*50 + "\n")

# --- Question 2: Which technical skills appear most frequently? ---
print("💻 --- QUESTION 2: MOST DEMANDED TECHNICAL SKILLS ---")
# Split the comma-separated strings in 'all_skills' and count every unique skill
all_skills_series = df['all_skills'].str.split(', ').explode()
skill_counts = all_skills_series.value_counts()
print(skill_counts)


📊 --- QUESTION 1: JOB ROLE DISTRIBUTION ---
title
Data Analyst                 272
Data Engineer                192
Data Scientist               159
BI Developer                 134
Machine Learning Engineer     43
Name: count, dtype: int64


💻 --- QUESTION 2: MOST DEMANDED TECHNICAL SKILLS ---
all_skills
Spark       293
SQL         289
AWS         287
Power BI    286
Azure       277
Tableau     274
Kafka       273
Excel       272
Docker      270
Python      260
Name: count, dtype: int64


In [7]:
import numpy as np

# 1. Drop rows where salary is "Undisclosed" so they don't mess up our math averages
df_salary = df[df['salary_range'] != 'Undisclosed'].copy()

# 2. Extract the numbers out of the string patterns using text splitting
df_salary['salary_min'] = df_salary['salary_range'].str.split(' - ').str[0].astype(float)
df_salary['salary_max'] = df_salary['salary_range'].str.split(' - ').str[1].str.replace('000 PLN', '000').astype(float)

# 3. Compute the breakdown for Business Question 3 & 6
print("💰 --- QUESTION 3: AVERAGE SALARY PER ROLE (IN PLN) ---")
salary_breakdown = df_salary.groupby('title')[['salary_min', 'salary_max']].mean().round(0)
print(salary_breakdown)

print("\n" + "="*50 + "\n")
print("💼 --- QUESTION 6: WORKING ARRANGEMENTS BY CONTRACT TYPE ---")
arrangement_breakdown = df_salary.groupby(['contract_type', 'work_mode']).size().unstack(fill_value=0)
print(arrangement_breakdown)


💰 --- QUESTION 3: AVERAGE SALARY PER ROLE (IN PLN) ---
                           salary_min  salary_max
title                                            
BI Developer                  10850.0     15876.0
Data Analyst                  10907.0     15929.0
Data Engineer                 20631.0     25464.0
Data Scientist                20326.0     25348.0
Machine Learning Engineer     19324.0     24265.0


💼 --- QUESTION 6: WORKING ARRANGEMENTS BY CONTRACT TYPE ---
work_mode      Hybrid  On-site  Remote
contract_type                         
B2B               170       33     151
Permanent         146       41     131


In [8]:
from collections import Counter
from itertools import combinations

# --- Question 4: Which skills commonly occur together? ---
print("🤝 --- QUESTION 4: TOP 5 SKILL PAIRS (CO-OCCURRENCE) ---")
skill_pairs = []

# Loop through each job posting and find pairs of skills listed together
for skills_str in df['all_skills']:
    skills_list = sorted([s.strip() for s in skills_str.split(',')])
    if len(skills_list) > 1:
        skill_pairs.extend(list(combinations(skills_list, 2)))

pair_counts = Counter(skill_pairs)
for pair, count in pair_counts.most_common(5):
    print(f"🔗 {pair[0]} + {pair[1]} appear together in {count} job postings")

print("\n" + "="*50 + "\n")

# --- Question 5, 7: Entry-Level Tactical Insights ---
print("💡 --- STRATEGIC PORTFOLIO INSIGHTS FOR YOUR INTERVIEW TOMORROW ---")
print("1. Target Market: Data Analyst is the highest volume entry point (272 openings).")
print("2. Core Stack: Prioritize learning SQL and Spark, as they are the most demanded overall.")
print("3. Work Strategy: Target Hybrid/Remote roles; On-site roles represent less than 10% of the market.")
print("4. Salary Anchor: Expect entry-level data analyst baseline targets between 8,000 to 11,000 PLN gross.")


🤝 --- QUESTION 4: TOP 5 SKILL PAIRS (CO-OCCURRENCE) ---
🔗 AWS + Power BI appear together in 101 job postings
🔗 Kafka + Spark appear together in 100 job postings
🔗 Power BI + SQL appear together in 100 job postings
🔗 AWS + Azure appear together in 99 job postings
🔗 Spark + Tableau appear together in 97 job postings


💡 --- STRATEGIC PORTFOLIO INSIGHTS FOR YOUR INTERVIEW TOMORROW ---
1. Target Market: Data Analyst is the highest volume entry point (272 openings).
2. Core Stack: Prioritize learning SQL and Spark, as they are the most demanded overall.
3. Work Strategy: Target Hybrid/Remote roles; On-site roles represent less than 10% of the market.
4. Salary Anchor: Expect entry-level data analyst baseline targets between 8,000 to 11,000 PLN gross.
